In [ ]:
import json
import cv2
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import ipywidgets as widgets
from matplotlib import animation
from IPython.display import HTML, Video, display
import tqdm as tqdm
from skimage.feature import local_binary_pattern
from scipy.spatial.distance import cdist
from scipy.spatial.distance import pdist
import os

In [ ]:
df = pd.read_csv("/home/guilherme_monteiro/projetos/tcc/data/metadata/video-metadata-publish-with-links.csv")
df['video_path'] = df['Filename'].apply(lambda x: "/home/guilherme_monteiro/projetos/tcc/data/videos/" + x)
df_true = df[df['Video Ground Truth'] == "Real"].reset_index(drop=True)
df_fake = df[df['Video Ground Truth'] == "Fake"].reset_index(drop=True)

def metadata_path_for_video(video_path):
    video_stem = os.path.splitext(os.path.basename(video_path))[0]
    return os.path.join(
        "/home/guilherme_monteiro/projetos/tcc/data/metadata",
        f"{video_stem}_meta.json"
    )

df['metadata_path'] = df['video_path'].apply(metadata_path_for_video)
df['has_metadata'] = df['metadata_path'].apply(os.path.exists)
df_meta = df[df['has_metadata']].reset_index(drop=True)

print(f"Vídeos no CSV: {len(df)}")
print(f"Vídeos com metadata disponível: {len(df_meta)}")
print(df_meta['Video Ground Truth'].value_counts())


# Funções básicas

In [ ]:
def get_video_frame_count(video_path):
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return frame_count


def sample_frame_indices(frame_count, max_frames=None):
    if frame_count <= 0:
        return np.array([], dtype=int)

    if max_frames is None or frame_count <= max_frames:
        return np.arange(frame_count, dtype=int)

    return np.linspace(0, frame_count - 1, int(max_frames)).astype(int)


def load_video_frames(video_path, max_frames=None, return_indices=False):
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = sample_frame_indices(frame_count, max_frames)

    frames = []
    used_indices = []

    for frame_idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ret, frame = cap.read()

        if not ret:
            continue

        frames.append(frame)
        used_indices.append(int(frame_idx))

    cap.release()

    frames = np.array(frames)

    if return_indices:
        return frames, np.array(used_indices, dtype=int), frame_count

    return frames


def standardize_frame(frame, max_size=640):
    h, w = frame.shape[:2]

    scale = min(max_size / max(h, w), 1.0)
    new_w, new_h = int(w * scale), int(h * scale)

    frame_resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)

    return frame_resized, scale


def load_metadata(video_path):
    full_path = metadata_path_for_video(video_path)

    with open(full_path, "r") as f:
        return json.load(f)


def metadata_index_for_frame(frame_idx, frame_count, metadata_len):
    if metadata_len <= 0:
        return None

    if metadata_len >= frame_count - 3:
        return min(int(frame_idx), metadata_len - 1)

    metadata_frame_indices = sample_frame_indices(frame_count, metadata_len)
    return int(np.argmin(np.abs(metadata_frame_indices - int(frame_idx))))


def scale_bbox(bbox, scale):
    return [int(round(x * scale)) for x in bbox]


def clip_bbox(bbox, width, height, min_size=4):
    x1, y1, x2, y2 = [int(round(v)) for v in bbox]

    x1 = max(0, min(width - 1, x1))
    x2 = max(0, min(width, x2))
    y1 = max(0, min(height - 1, y1))
    y2 = max(0, min(height, y2))

    if x2 <= x1:
        x2 = min(width, x1 + min_size)
    if y2 <= y1:
        y2 = min(height, y1 + min_size)

    return [x1, y1, x2, y2]


def create_face_regions(frame, bbox, padding=0.2):
    h, w = frame.shape[:2]

    x1, y1, x2, y2 = clip_bbox(bbox, w, h)

    bw = x2 - x1
    bh = y2 - y1

    px = int(bw * padding)
    py = int(bh * padding)

    x1p = max(0, x1 - px)
    y1p = max(0, y1 - py)
    x2p = min(w, x2 + px)
    y2p = min(h, y2 + py)

    face_mask = np.zeros((h, w), dtype=np.uint8)
    face_mask[y1:y2, x1:x2] = 1

    expanded_mask = np.zeros((h, w), dtype=np.uint8)
    expanded_mask[y1p:y2p, x1p:x2p] = 1

    border_mask = expanded_mask - face_mask
    background_mask = 1 - expanded_mask

    return {
        "face": face_mask,
        "border": border_mask,
        "background": background_mask,
        "bbox": (x1, y1, x2, y2),
        "bbox_expanded": (x1p, y1p, x2p, y2p)
    }


## Funções pra visualização

In [ ]:
def draw_text_block(img, text_lines, x=10, y=20, line_height=18):
    for i, line in enumerate(text_lines):
        cv2.putText(
            img,
            line,
            (x, y + i * line_height),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 255, 255),
            1,
            cv2.LINE_AA
        )
    return img

# Desenha a caixa delimitadora no frame
def draw_face_box(frame, bbox, color=(0, 255, 0), thickness=2):
    x1, y1, x2, y2 = bbox
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    return frame


def overlay_regions(frame, regions):
    overlay = frame.copy()

    face_mask = regions["face"]
    border_mask = regions["border"]
    background_mask = regions["background"]

    overlay[face_mask == 1] = (0, 255, 0)       # verde
    overlay[border_mask == 1] = (0, 0, 255)     # vermelho
    overlay[background_mask == 1] = (255, 0, 0) # azul

    alpha = 0.3
    return cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)

# Ruido Residual

In [ ]:
def compute_residual(gray, method="bilateral", kernel_size=9, sigma_color=75, sigma_space=75):
    if gray.ndim == 3:
        gray = cv2.cvtColor(gray, cv2.COLOR_BGR2GRAY)

    gray_float = gray.astype(np.float32)

    if method == "bilateral":
        smooth = cv2.bilateralFilter(gray, kernel_size, sigma_color, sigma_space).astype(np.float32)
    elif method == "median":
        smooth = cv2.medianBlur(gray, kernel_size).astype(np.float32)
    else:
        raise ValueError(f"Método de residual não suportado: {method}")

    residual = gray_float - smooth
    return residual


In [ ]:
def compute_noise_region(residual, mask):
    values = residual[mask == 1].astype(np.float32)

    if len(values) < 25:
        return None

    mean = float(np.mean(values))
    variance = float(np.var(values))
    std = float(np.sqrt(variance))
    energy = float(np.mean(np.abs(values)))
    rms = float(np.sqrt(np.mean(values ** 2)))
    median = float(np.median(values))
    mad = float(np.median(np.abs(values - median)))
    p95_abs = float(np.percentile(np.abs(values), 95))

    hist, _ = np.histogram(values, bins=64, density=False)
    hist = hist.astype(np.float64)
    hist = hist / (np.sum(hist) + 1e-12)
    entropy = float(-np.sum(hist * np.log(hist + 1e-12)) / np.log(len(hist)))

    centered = values - mean
    if std < 1e-6:
        skewness = 0.0
        kurtosis = 0.0
    else:
        skewness = float(np.mean(centered ** 3) / (std ** 3))
        kurtosis = float(np.mean(centered ** 4) / (variance ** 2 + 1e-12))

    return {
        "mean": mean,
        "variance": variance,
        "std": std,
        "energy": energy,
        "rms": rms,
        "entropy_norm": entropy,
        "skewness": skewness,
        "kurtosis": kurtosis,
        "mad": mad,
        "p95_abs": p95_abs,
    }


def noise_region_differences(f1, f2, prefix):
    if f1 is None or f2 is None:
        return {}

    comparable_metrics = [
        "variance",
        "std",
        "energy",
        "rms",
        "entropy_norm",
        "skewness",
        "kurtosis",
        "mad",
        "p95_abs",
    ]

    return {
        f"{prefix}_noise_{metric}_diff": abs(f1[metric] - f2[metric])
        for metric in comparable_metrics
    }


def compute_noise_metrics(frame, bbox):
    frame_std, scale = standardize_frame(frame)
    bbox_scaled = scale_bbox(bbox, scale)
    bbox_scaled = clip_bbox(bbox_scaled, frame_std.shape[1], frame_std.shape[0])

    gray = cv2.cvtColor(frame_std, cv2.COLOR_BGR2GRAY)
    residual = compute_residual(gray)

    regions = create_face_regions(frame_std, bbox_scaled)

    face = compute_noise_region(residual, regions["face"])
    border = compute_noise_region(residual, regions["border"])
    bg = compute_noise_region(residual, regions["background"])

    features = {}

    for region_name, region_features in [("face", face), ("border", border), ("background", bg)]:
        if region_features is None:
            continue

        for metric_name, metric_value in region_features.items():
            features[f"noise_{region_name}_{metric_name}"] = metric_value

    features.update(noise_region_differences(face, bg, "face_bg"))
    features.update(noise_region_differences(face, border, "face_border"))
    features.update(noise_region_differences(border, bg, "border_bg"))

    return features, residual


## Debbug por video

In [ ]:
def residual_visual(residual):
    res = residual.copy()
    res = np.clip(res * 3, -255, 255)
    res_abs = np.abs(res)
    res_vis = cv2.normalize(res_abs, None, 0, 255, cv2.NORM_MINMAX)
    return res_vis.astype(np.uint8)


def debug_noise_video(video_path, max_frames=120, interval=60):
    frames, frame_indices, frame_count = load_video_frames(video_path, max_frames=max_frames, return_indices=True)
    metadata = load_metadata(video_path)

    debug_frames = []

    print("\nProcessando Residual Noise...")

    for i, frame in enumerate(tqdm.tqdm(frames)):
        frame_idx = int(frame_indices[i])
        metadata_idx = metadata_index_for_frame(frame_idx, frame_count, len(metadata))

        if metadata_idx is None:
            continue

        frame_std, scale = standardize_frame(frame)
        bbox = metadata[metadata_idx]["bbox"]
        bbox_scaled = scale_bbox(bbox, scale)
        bbox_scaled = clip_bbox(bbox_scaled, frame_std.shape[1], frame_std.shape[0])

        regions = create_face_regions(frame_std, bbox_scaled)
        features, residual = compute_noise_metrics(frame, bbox)
        res_vis = residual_visual(residual)

        overlay = overlay_regions(frame_std.copy(), regions)
        overlay = draw_face_box(overlay, bbox_scaled)

        text_lines = [
            f"Frame: {frame_idx} | Meta: {metadata_idx}",
            f"Face Var: {features.get('noise_face_variance', 0):.4f}",
            f"Face Energy: {features.get('noise_face_energy', 0):.4f}",
            f"Face Entropy: {features.get('noise_face_entropy_norm', 0):.3f}",
            f"Face-BG Var Diff: {features.get('face_bg_noise_variance_diff', 0):.4f}",
        ]

        overlay = draw_text_block(overlay, text_lines)

        combined = np.hstack([
            cv2.cvtColor(frame_std, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(res_vis, cv2.COLOR_GRAY2RGB)
        ])

        debug_frames.append(combined)

    if not debug_frames:
        raise ValueError("Nenhum frame válido para visualização")

    print("\nRenderizando...")

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.axis("off")

    img = ax.imshow(debug_frames[0])

    def update(i):
        img.set_data(debug_frames[i])
        return [img]

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(debug_frames),
        interval=interval,
        blit=True,
    )

    plt.close(fig)
    display(HTML(ani.to_jshtml()))


In [ ]:
#debug_noise_video(df_true.iloc[1]["video_path"], max_frames=200)

In [ ]:
#debug_noise_video(df_fake.iloc[2]["video_path"], max_frames=200)

In [ ]:
#debug_noise_video(df_fake.iloc[1]["video_path"], max_frames=200)

# Todas as metricas

In [ ]:
def all_metrics(video_path, max_frames=500, label=None):
    video_name = os.path.basename(video_path)
    print(f"\nExtraindo métricas do vídeo: {video_name}")

    frames, frame_indices, frame_count = load_video_frames(video_path, max_frames=max_frames, return_indices=True)
    metadata = load_metadata(video_path)

    all_features = []

    for i, frame in enumerate(frames):
        frame_idx = int(frame_indices[i])
        metadata_idx = metadata_index_for_frame(frame_idx, frame_count, len(metadata))

        if metadata_idx is None:
            continue

        bbox = metadata[metadata_idx]["bbox"]
        features_noise, _ = compute_noise_metrics(frame, bbox)

        features = {**features_noise}
        features["frame"] = frame_idx
        features["metadata_idx"] = metadata_idx

        all_features.append(features)

    metrics_df = pd.DataFrame(all_features)
    metrics_df.insert(0, "video_name", video_name)

    if label is not None:
        metrics_df.insert(1, "label", label)

    return metrics_df


def metadata_exists(video_path):
    return os.path.exists(metadata_path_for_video(video_path))


def aggregate_video_metrics(frame_metrics):
    metric_cols = [
        col for col in frame_metrics.columns
        if col.startswith("noise_") or "_noise_" in col
    ]

    agg = frame_metrics[metric_cols].agg(["mean", "std", "median"])
    values = {}

    for metric in metric_cols:
        values[f"{metric}_mean"] = agg.loc["mean", metric]
        values[f"{metric}_std"] = agg.loc["std", metric]
        values[f"{metric}_median"] = agg.loc["median", metric]

    return values


def auc_from_scores(labels, scores, positive_label="Fake"):
    valid = [(label, score) for label, score in zip(labels, scores) if pd.notna(score)]
    pos = [score for label, score in valid if label == positive_label]
    neg = [score for label, score in valid if label != positive_label]

    if len(pos) == 0 or len(neg) == 0:
        return np.nan

    wins = 0
    ties = 0

    for pos_score in pos:
        for neg_score in neg:
            wins += pos_score > neg_score
            ties += pos_score == neg_score

    return (wins + 0.5 * ties) / (len(pos) * len(neg))


def cohen_d_by_label(values, labels, positive_label="Fake"):
    values = pd.Series(values)
    labels = pd.Series(labels)

    pos = values[labels == positive_label].dropna().astype(float)
    neg = values[labels != positive_label].dropna().astype(float)

    if len(pos) < 2 or len(neg) < 2:
        return np.nan

    pooled = np.sqrt((pos.var(ddof=1) + neg.var(ddof=1)) / 2)

    if pooled == 0 or pd.isna(pooled):
        return np.nan

    return (pos.mean() - neg.mean()) / pooled


def evaluate_group_c(max_frames=120):
    rows = []
    available_df = df[df['video_path'].apply(metadata_exists)].reset_index(drop=True)

    for _, row in available_df.iterrows():
        frame_metrics = all_metrics(
            row['video_path'],
            max_frames=max_frames,
            label=row['Video Ground Truth']
        )

        if frame_metrics.empty:
            continue

        video_row = aggregate_video_metrics(frame_metrics)
        video_row["video_name"] = os.path.basename(row['video_path'])
        video_row["label"] = row['Video Ground Truth']
        video_row["n_frames"] = len(frame_metrics)
        rows.append(video_row)

    video_metrics = pd.DataFrame(rows)

    metric_cols = [
        col for col in video_metrics.columns
        if (col.startswith("noise_") or "_noise_" in col) and col.endswith("_mean")
    ]

    report_rows = []
    for metric in metric_cols:
        auc = auc_from_scores(video_metrics['label'], video_metrics[metric])
        auc_abs = max(auc, 1 - auc) if pd.notna(auc) else np.nan
        d = cohen_d_by_label(video_metrics[metric], video_metrics['label'])

        report_rows.append({
            "metric": metric,
            "auc_fake": auc,
            "auc_abs": auc_abs,
            "cohen_d_fake_minus_real": d,
            "real_mean": video_metrics.loc[video_metrics['label'] == "Real", metric].mean(),
            "fake_mean": video_metrics.loc[video_metrics['label'] == "Fake", metric].mean(),
        })

    report = pd.DataFrame(report_rows)

    if not report.empty:
        report = report.sort_values(
            ["auc_abs", "cohen_d_fake_minus_real"],
            ascending=[False, False],
            key=lambda s: s.abs() if s.name == "cohen_d_fake_minus_real" else s,
        ).reset_index(drop=True)

    print("\nVídeos avaliados:")
    print(video_metrics['label'].value_counts())

    return video_metrics, report


## Avaliação discriminativa

Executa a extração apenas nos vídeos com metadata disponível e resume uma linha por vídeo. Use `report.head(20)` para ver quais sinais de ruído residual separam melhor Real/Fake nesta amostra.

In [ ]:
# Rodada rápida de validação. Aumente max_frames para estabilizar os resultados.
video_metrics, report = evaluate_group_c(max_frames=120)
display(report.head(20))